# Automatic Object Removal — keep the main person, erase the rest
**Computer Vision — Sapienza Università di Roma — A.A. 2025–2026**

Omri Ben Zion & Johannes Erhard · improved Kaggle pipeline

**Goal:** fully automatic *object removal* on our own Rome photos. Detect every person, keep the
main subject, and **diffuse the background into the others' place** (not a new person).

**Pipeline:** Mask R-CNN (find all people) → pick main subject → SAM (refine removal masks) →
BLIP caption (background) → **OSRG** inpainting → composite back at full resolution.

**Novelty — OSRG (Object-Suppressing Removal Guidance):** a spatially-masked, object-conditioned
guidance term that, *only inside the hole*, steers the diffusion away from the removed object's
semantics and toward the background prior. We compare it against the SD/RePaint baselines and a
global negative-prompt ablation to prove the spatial masking is the cause of the improvement.

**Code structure (mandatory order):** §1 Imports → §2 Globals → §3 Utils → §4 Data → §5 Network → §6 Pipeline → §7 Evaluation

## § 1 · Imports

In [ ]:
# Kaggle install — install a torch build whose CUDA kernels match the GPU.
import sys, subprocess
def _pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *a], capture_output=True, text=True)
    print("pip rc", r.returncode, "|", (r.stderr.strip()[-200:] or "ok"))
_pip("-q", "torch==2.5.1", "torchvision==0.20.1",
     "--index-url", "https://download.pytorch.org/whl/cu121")
with open("/tmp/c.txt", "w") as f:
    f.write("torch==2.5.1\ntorchvision==0.20.1\n")
_pip("-q", "-U", "diffusers>=0.31.0", "transformers>=4.45.0",
     "accelerate>=0.34.0", "safetensors", "lpips", "pycocotools", "-c", "/tmp/c.txt")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| pip done")

In [ ]:
import os, csv, json, gc, math, time, random, zipfile
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib.pyplot as plt

from transformers import SamModel, SamProcessor, BlipProcessor, BlipForConditionalGeneration
from diffusers import StableDiffusionInpaintPipeline

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## § 2 · Globals

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# P100 has half-rate fp16 -> fp16 inference is pathologically slow there; use fp32.
DTYPE = torch.float32 if ("P100" in GPU_NAME or DEVICE == "cpu") else torch.float16
print("Device:", DEVICE, "| GPU:", GPU_NAME, "| dtype:", DTYPE)

# paths
INPUT_DIR = Path("/kaggle/input/cv-rome-removal-photos")
OUT_DIR   = Path("/kaggle/working/steps"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# diffusion working resolution (3:2, multiples of 8)
PROC_W, PROC_H   = 768, 512
NUM_STEPS        = 30          # P100 is slow; 30 is enough for background fill
GUIDANCE         = 6.0          # low-ish: lean on context, do not invent a subject
N_STEP_SNAPSHOTS = 6            # decoded denoising frames saved for the slides

# removal-mask shaping
DILATE_PX         = 22          # generous: covers edges/halos of removed people
DILATE_DOWN_EXTRA = 16          # extra downward growth → catches shadow / contact
FEATHER_PX        = 11          # soft composite seam

DETECT_SCORE_THR  = 0.75
MIN_REMOVE_FRAC   = 0.0015      # ignore specks
MAIN_MIN_FRAC     = 0.03        # a "main subject" must occupy at least this area

# prompts
PERSON_NEG = ("person, people, man, woman, boy, girl, child, human, tourist, crowd, "
              "face, body, figure, silhouette, pedestrian, ")
BASE_NEG   = "blurry, distorted, low quality, watermark, text, deformed, artifacts, extra limbs"
REMOVAL_NEG = PERSON_NEG + BASE_NEG
BG_PROMPT_TMPL = "empty {bg}, clean continuous background, no people, photorealistic, natural daylight"
NAIVE_PROMPT   = "a photo"      # deliberately generic → shows the re-insertion failure

# OSRG (our novelty)
OSRG_SCALE = 3.0                # object-suppression guidance strength s
OBJ_PROMPT = "a person, people, a man, a woman, a human figure"   # what to suppress

# quantitative ablation benchmark
BENCH_N = 16                    # synthetic copy-paste removal pairs (true GT)

SEED = 42

# per-photo control
PHOTO_CONFIG = {
    "DSCF5883": {"keep_point": (905, 470),  "protect_points": []},  # Trevi — sunglasses guy center
    "DSCF5909": {"keep_point": (880, 520),  "protect_points": []},  # Navona — white-T guy center-right
    "DSCF5905": {"keep_point": (888, 470),  "protect_points": []},  # Navona — white-T guy center
    "DSCF5873": {"keep_point": (905, 460),  "protect_points": []},  # Trevi — sunglasses guy (alone)
    "DSCF5878": {"keep_point": (888, 560),  "protect_points": []},  # Trevi — curly white-T guy center
    "DSCF5908": {"keep_point": (875, 600),  "protect_points": []},  # Navona — white-T guy; blue-cap man left removed
}
print("Globals set.")

## § 3 · Utils

In [ ]:
def set_all_seeds(seed: int) -> None:
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)

def _gen(seed: int):
    g = torch.Generator(device=DEVICE); g.manual_seed(seed); return g

def _font(size=20):
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/opt/conda/fonts/DejaVuSans.ttf"]:
        try: return ImageFont.truetype(p, size)
        except Exception: pass
    return ImageFont.load_default()

In [ ]:
# Detection: every person via Mask R-CNN
PERSON_CLASS = 1

def detect_persons(image: Image.Image, thr: float = DETECT_SCORE_THR) -> list:
    W, H = image.size
    t = TF.to_tensor(image.convert("RGB")).to(DEVICE)
    with torch.no_grad():
        out = detector([t])[0]
    persons = []
    for box, lbl, score, mask in zip(out["boxes"], out["labels"], out["scores"], out["masks"]):
        if int(lbl) != PERSON_CLASS or float(score) < thr:
            continue
        m = (mask[0] > 0.5).cpu().numpy().astype(np.uint8) * 255   # 0/255, not 0/1
        area = int((m > 0).sum())
        if area / float(W * H) < MIN_REMOVE_FRAC:
            continue
        ys, xs = np.where(m > 0)
        persons.append({
            "box": box.cpu().numpy().astype(int), "score": float(score),
            "mask": m, "area": area, "frac": area / float(W * H),
            "cx": float(xs.mean()), "cy": float(ys.mean()),
        })
    persons.sort(key=lambda p: p["area"], reverse=True)
    return persons

def select_main(persons: list, W: int, H: int, cfg: dict):
    if not persons:
        return None
    kp = cfg.get("keep_point")
    if kp is not None:
        # the detected person whose mask contains the keep point, else nearest centroid
        inside = [i for i, p in enumerate(persons) if p["mask"][min(H-1, kp[1]), min(W-1, kp[0])] > 0]
        if inside:
            return max(inside, key=lambda i: persons[i]["area"])
        return min(range(len(persons)),
                   key=lambda i: (persons[i]["cx"] - kp[0])**2 + (persons[i]["cy"] - kp[1])**2)
    # fallback: most central among sufficiently large persons
    big = [i for i, p in enumerate(persons) if p["frac"] >= MAIN_MIN_FRAC] or list(range(len(persons)))
    return min(big, key=lambda i: abs(persons[i]["cx"] / W - 0.5))

def _protected(p, cfg, W, H):
    for (px, py) in cfg.get("protect_points", []):
        if p["mask"][min(H-1, py), min(W-1, px)] > 0:
            return True
    return False

In [ ]:
# SAM mask refinement (box-prompted)
def sam_refine_box(image: Image.Image, box) -> np.ndarray:
    img_rgb = np.array(image.convert("RGB"))
    inputs = sam_processor(images=img_rgb,
                           input_boxes=[[[float(box[0]), float(box[1]),
                                          float(box[2]), float(box[3])]]],
                           return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = sam_model(**inputs)
    masks = sam_processor.post_process_masks(out.pred_masks.cpu(),
                                             inputs["original_sizes"].cpu(),
                                             inputs["reshaped_input_sizes"].cpu())
    scores = out.iou_scores[0, 0]
    best = int(scores.argmax())
    return masks[0][0][best].numpy().astype(np.uint8) * 255   # 0/255

def build_removal_mask(image, persons, main_idx, cfg, use_sam=True):
    W, H = image.size
    raw = np.zeros((H, W), np.uint8)
    refined = np.zeros((H, W), np.uint8)
    removed = []
    for i, p in enumerate(persons):
        if i == main_idx or _protected(p, cfg, W, H):
            continue
        raw |= p["mask"]
        if use_sam:
            try:
                refined |= sam_refine_box(image, p["box"])
            except Exception as e:
                print("  [SAM warn]", e); refined |= p["mask"]
        else:
            refined |= p["mask"]
        removed.append(i)
    return raw, refined, removed

def dilate_mask(mask: np.ndarray, px: int = DILATE_PX, down_extra: int = DILATE_DOWN_EXTRA):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*px+1, 2*px+1))
    base = cv2.dilate(mask, k)
    if down_extra > 0:                       # bias the growth downward (shadows)
        shifted = np.zeros_like(base)
        shifted[down_extra:, :] = base[:-down_extra, :]
        base = np.maximum(base, shifted)
    return base

def blackout_persons(image, persons):
    img = np.array(image.convert("RGB")).copy()
    allm = np.zeros(img.shape[:2], np.uint8)
    for p in persons: allm |= p["mask"]
    allm = dilate_mask(allm, 6, 0)
    img[allm > 0] = 127
    return Image.fromarray(img)

In [ ]:
# BLIP background captioning
_BAD = ("black", "blank", "gray", "grey", "dark")
def caption_background(image_no_people: Image.Image) -> str:
    inputs = blip_processor(images=image_no_people, return_tensors="pt").to(DEVICE, DTYPE)
    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_new_tokens=30)
    cap = blip_processor.decode(ids[0], skip_special_tokens=True).strip()
    if not cap or any(w in cap.lower() for w in _BAD):
        cap = "a street scene with historic architecture"
    return cap

# classical pre-fill (background-biased)
def telea_prefill(image_small: Image.Image, mask_small: Image.Image) -> Image.Image:
    bgr = cv2.cvtColor(np.array(image_small.convert("RGB")), cv2.COLOR_RGB2BGR)
    m = (np.array(mask_small.convert("L")) > 127).astype(np.uint8) * 255
    filled = cv2.inpaint(bgr, m, 5, cv2.INPAINT_TELEA)
    return Image.fromarray(cv2.cvtColor(filled, cv2.COLOR_BGR2RGB))

In [ ]:
# Denoising-step capture (for the slides)
def make_step_callback(store, total, n_snap=N_STEP_SNAPSHOTS):
    want = set(np.linspace(0, total - 1, n_snap).astype(int).tolist())
    def cb(pipe, step, timestep, kw):
        if step in want:
            lat = kw["latents"]
            with torch.no_grad():
                img = pipe.vae.decode(lat / pipe.vae.config.scaling_factor, return_dict=False)[0]
            img = (img / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).float().cpu().numpy()
            store.append((step, Image.fromarray((img * 255).astype(np.uint8))))
        return kw
    return cb

In [ ]:
# Visualization helpers
def draw_detections(image, persons, main_idx):
    im = image.convert("RGB").copy(); d = ImageDraw.Draw(im); f = _font(28)
    for i, p in enumerate(persons):
        x0, y0, x1, y1 = p["box"]
        keep = (i == main_idx)
        col = (40, 200, 60) if keep else (235, 40, 40)
        d.rectangle([x0, y0, x1, y1], outline=col, width=5)
        tag = f"KEEP #{i}" if keep else f"remove #{i}"
        d.text((x0 + 4, max(0, y0 - 30)), f"{tag} {p['score']:.2f}", fill=col, font=f)
    return im

def mask_overlay(image, mask, color=(235, 40, 40), alpha=0.5):
    im = np.array(image.convert("RGB")).astype(np.float32)
    m = (np.array(Image.fromarray(mask).resize(image.size, Image.NEAREST)) > 127) \
        if mask.shape[:2] != image.size[::-1] else (mask > 127)
    ov = im.copy()
    for c in range(3): ov[..., c] = np.where(m, (1-alpha)*im[..., c] + alpha*color[c], im[..., c])
    return Image.fromarray(ov.astype(np.uint8))

def label_img(img, text, h=34):
    img = img.convert("RGB"); w = img.width
    out = Image.new("RGB", (w, img.height + h), (255, 255, 255))
    out.paste(img, (0, h)); d = ImageDraw.Draw(out); f = _font(22)
    d.text((6, 6), text, fill=(0, 0, 0), font=f); return out

def hstrip(pairs, cell_w=384):
    imgs = [label_img(im.resize((cell_w, int(cell_w*im.height/im.width))), lab) for lab, im in pairs]
    H = max(i.height for i in imgs); W = sum(i.width for i in imgs) + 6*(len(imgs)-1)
    canvas = Image.new("RGB", (W, H), (255, 255, 255)); x = 0
    for i in imgs: canvas.paste(i, (x, 0)); x += i.width + 6
    return canvas

def grid(pairs, cols=3, cell_w=440):
    cells = [label_img(im.resize((cell_w, int(cell_w*im.height/im.width))), lab) for lab, im in pairs]
    rows = math.ceil(len(cells)/cols); cw = cell_w; ch = max(c.height for c in cells)
    canvas = Image.new("RGB", (cols*cw + 8*(cols-1), rows*ch + 8*(rows-1)), (255, 255, 255))
    for k, c in enumerate(cells):
        r, cc = divmod(k, cols); canvas.paste(c, (cc*(cw+8), r*(ch+8)))
    return canvas

def reinsertion(result_img, mask_np, exclude_np=None, thr=0.5, overlap=0.10):
    # 1 if a person is still detected inside the removed region (failed removal).
    m = mask_np > 127; tot = max(1, int(m.sum()))
    ex = (exclude_np > 127) if exclude_np is not None else None
    for p in detect_persons(result_img, thr=thr):
        pm = p["mask"] > 0
        if (pm & m).sum() / tot <= overlap:
            continue
        if ex is not None and (pm & ex).sum() / max(1, int(pm.sum())) > 0.5:
            continue  # this detection is the kept main subject, not a re-insertion
        return 1
    return 0

def mask_bbox(mask_np, pad=14):
    m = mask_np > 127
    ys, xs = np.where(m)
    if len(xs) == 0: return (0, 0, mask_np.shape[1], mask_np.shape[0])
    H, W = mask_np.shape[:2]
    return (max(0, int(xs.min())-pad), max(0, int(ys.min())-pad),
            min(W, int(xs.max())+pad), min(H, int(ys.max())+pad))

def composite_back(original_full, result_small, mask_full):
    res_up = result_small.resize(original_full.size, Image.LANCZOS)
    a = Image.fromarray(mask_full).resize(original_full.size, Image.NEAREST)
    a = a.filter(ImageFilter.GaussianBlur(FEATHER_PX))
    a = np.array(a).astype(np.float32) / 255.0
    o = np.array(original_full.convert("RGB")).astype(np.float32)
    r = np.array(res_up.convert("RGB")).astype(np.float32)
    out = o * (1 - a[..., None]) + r * a[..., None]
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))

## § 4 · Data

In [ ]:
def _find_input_dir() -> Path:
    if INPUT_DIR.exists():
        return INPUT_DIR
    root = Path("/kaggle/input")
    print("INPUT_DIR missing; /kaggle/input contains:",
          [p.name for p in root.iterdir()] if root.exists() else "<none>")
    # any folder under /kaggle/input that holds our DSCF photos
    for p in root.rglob("DSCF*.jpg"):
        return p.parent
    return INPUT_DIR

def load_photos() -> list:
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    src = _find_input_dir()
    samples = []
    for p in sorted(src.iterdir()):
        if p.suffix.lower() not in exts: continue
        img = Image.open(p).convert("RGB")
        samples.append({"image": img, "stem": p.stem, "path": str(p)})
    print(f"Loaded {len(samples)} photos from {src}")
    for s in samples: print(" -", s["stem"], s["image"].size)
    return samples

## § 5 · Network

In [ ]:
# Load all models once. Order: detector → SAM → BLIP → SD-inpaint.
print("Loading Mask R-CNN detector ...")
detector = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT).eval().to(DEVICE)

print("Loading SAM ...")
sam_model = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE).eval()
sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

print("Loading BLIP captioner ...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large", torch_dtype=DTYPE).to(DEVICE).eval()

print("Loading SD inpainting ...")
# Must use checkpoints that ship .safetensors — diffusers refuses .bin pickles
SD_INPAINT_IDS = ["Lykon/dreamshaper-8-inpainting",            # SD1.5, safetensors
                  "stabilityai/stable-diffusion-2-inpainting"] # safetensors fallback
inpaint_pipe = None
for mid in SD_INPAINT_IDS:
    try:
        inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, safety_checker=None, use_safetensors=True).to(DEVICE)
        inpaint_pipe.set_progress_bar_config(disable=True)
        print("  loaded", mid); break
    except Exception as e:
        print("  [warn]", mid, "->", str(e)[:160])
assert inpaint_pipe is not None, "no SD-inpaint checkpoint loaded"
if DEVICE == "cuda":
    print("VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")
print("All models loaded.")

## § 6 · Pipeline
> *No model training in this project.* This section assembles the full automatic
> object-removal pipeline and runs both a **naive** pass (re-inserts a subject) and our
> **removal-tuned** pass (diffuses the background back in), dumping every step for the slides.

In [ ]:
def inpaint(image_small, mask_small, prompt, neg, capture=False, seed=SEED):
    set_all_seeds(seed); g = _gen(seed)
    steps = []
    cb = make_step_callback(steps, NUM_STEPS) if capture else None
    kw = dict(prompt=prompt, negative_prompt=neg, image=image_small, mask_image=mask_small,
              num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=g,
              width=PROC_W, height=PROC_H)
    if cb is not None:
        kw["callback_on_step_end"] = cb
        kw["callback_on_step_end_tensor_inputs"] = ["latents"]
    res = inpaint_pipe(**kw).images[0]
    return res, steps

# OSRG: Object-Suppressing Removal Guidance
@torch.no_grad()
def osrg_inpaint(image_small, mask_small, bg_prompt, obj_prompt,
                 osrg_scale=OSRG_SCALE, guidance=GUIDANCE, steps=NUM_STEPS,
                 seed=SEED, capture=False):
    pipe, dev = inpaint_pipe, DEVICE
    g = _gen(seed)

    def embed(p):
        ids = pipe.tokenizer(p, padding="max_length", truncation=True,
                             max_length=pipe.tokenizer.model_max_length,
                             return_tensors="pt").input_ids.to(dev)
        return pipe.text_encoder(ids)[0]
    emb = torch.cat([embed(""), embed(bg_prompt), embed(obj_prompt)], 0).to(DTYPE)

    img = np.asarray(image_small.convert("RGB"), np.float32) / 255.0
    img = (torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(dev, DTYPE) * 2 - 1)
    m = (np.asarray(mask_small.convert("L"), np.float32) / 255.0 > 0.5).astype(np.float32)
    m = torch.from_numpy(m).unsqueeze(0).unsqueeze(0).to(dev, DTYPE)          # 1=hole
    sf = pipe.vae.config.scaling_factor
    masked_lat = pipe.vae.encode(img * (m < 0.5)).latent_dist.mode() * sf
    h, w = masked_lat.shape[-2:]
    mask_lat = F.interpolate(m, size=(h, w), mode="nearest")

    pipe.scheduler.set_timesteps(steps, device=dev)
    lat = torch.randn((1, 4, h, w), generator=g, device=dev, dtype=DTYPE) \
        * pipe.scheduler.init_noise_sigma

    snaps, want = [], set(np.linspace(0, steps - 1, N_STEP_SNAPSHOTS).astype(int).tolist())
    def _decode(x):
        d = pipe.vae.decode(x / sf, return_dict=False)[0]
        d = (d / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).float().cpu().numpy()
        return Image.fromarray((d * 255).astype(np.uint8))

    for i, t in enumerate(pipe.scheduler.timesteps):
        li = pipe.scheduler.scale_model_input(lat, t)
        model_in = torch.cat([torch.cat([li] * 3),
                              torch.cat([mask_lat] * 3),
                              torch.cat([masked_lat] * 3)], dim=1)        # (3,9,h,w)
        n_unc, n_bg, n_obj = pipe.unet(model_in, t, encoder_hidden_states=emb).sample.chunk(3)
        n_cfg = n_unc + guidance * (n_bg - n_unc)                          # toward background
        n_sup = n_cfg + osrg_scale * (n_bg - n_obj)                        # suppress object
        n_final = n_cfg * (1 - mask_lat) + n_sup * mask_lat                # only in the hole
        lat = pipe.scheduler.step(n_final, t, lat).prev_sample
        if capture and i in want:
            snaps.append((i, _decode(lat)))
    return _decode(lat), snaps

def run_removal(sample: dict) -> dict:
    stem = sample["stem"]; orig = sample["image"]; W, H = orig.size
    cfg = PHOTO_CONFIG.get(stem, {})
    d = OUT_DIR / stem; d.mkdir(parents=True, exist_ok=True)
    def save(img, name): img.save(d / name); return img
    t0 = time.perf_counter(); times = {}

    # 1) detect every person
    _t = time.perf_counter()
    persons = detect_persons(orig)
    times["detect"] = time.perf_counter() - _t
    main_idx = select_main(persons, W, H, cfg)
    print(f"[{stem}] {len(persons)} person(s), main=#{main_idx}")
    save(orig, "00_original.png")
    save(draw_detections(orig, persons, main_idx), "01_detections.png")

    # 2) build removal mask (everyone except main) + SAM refine
    _t = time.perf_counter()
    raw, refined, removed = build_removal_mask(orig, persons, main_idx, cfg, use_sam=True)
    times["sam_mask"] = time.perf_counter() - _t
    if refined.sum() == 0:
        print(f"  nothing to remove for {stem} (clean main-subject shot).")
        save(orig, "11_removal_result.png")
        save(hstrip([("original / kept as-is", orig)]), "99_storyboard.png")
        return {"stem": stem, "removed": 0, "result": orig}

    dil = dilate_mask(refined)
    save(mask_overlay(orig, refined), "03_mask_refined.png")
    save(mask_overlay(orig, dil),     "05_mask_dilated.png")
    masked_in = np.array(orig).copy(); masked_in[dil > 0] = 255
    save(Image.fromarray(masked_in), "06_masked_input.png")

    # 3) background caption (people blacked out)
    no_people = save(blackout_persons(orig, persons), "07_people_blacked.png")
    _t = time.perf_counter()
    bg = caption_background(no_people)
    times["caption"] = time.perf_counter() - _t
    prompt = BG_PROMPT_TMPL.format(bg=bg)
    (d / "08_caption.txt").write_text(f"BLIP background: {bg}\nPrompt: {prompt}\nNeg: {REMOVAL_NEG}")
    print("  caption:", bg)

    # 4) work at diffusion resolution
    img_s  = orig.resize((PROC_W, PROC_H), Image.LANCZOS)
    mask_s = Image.fromarray(dil).resize((PROC_W, PROC_H), Image.NEAREST)
    tight_s = Image.fromarray(refined).resize((PROC_W, PROC_H), Image.NEAREST)

    pre = save(telea_prefill(img_s, mask_s), "09_telea_prefill.png")

    # 5a) NAIVE pass — generic prompt, tight mask, no negative → typically re-inserts a subject
    naive_s, _ = inpaint(img_s, tight_s, NAIVE_PROMPT, "")
    naive_full = composite_back(orig, naive_s, refined)
    save(naive_full, "10_naive_result.png")

    # 5b) TUNED removal pass — bg prompt + person-negative + dilated mask + step capture
    tuned_s, steps = inpaint(img_s, mask_s, prompt, REMOVAL_NEG, capture=True)
    tuned_full = composite_back(orig, tuned_s, dil)
    save(tuned_full, "11_removal_result.png")

    if steps:
        save(hstrip([(f"step {s}/{NUM_STEPS}", im) for s, im in steps], cell_w=300),
             "12_denoise_steps.png")

    # 5c) OSRG — Object-Suppressing Removal Guidance , spatially masked
    _t = time.perf_counter()
    osrg_s, osrg_steps = osrg_inpaint(img_s, mask_s, prompt, OBJ_PROMPT, capture=True)
    times["inpaint_osrg"] = time.perf_counter() - _t
    osrg_full = composite_back(orig, osrg_s, dil)
    save(osrg_full, "14_osrg_result.png")
    if osrg_steps:
        save(hstrip([(f"step {i}/{NUM_STEPS}", im) for i, im in osrg_steps], cell_w=300),
             "15_osrg_denoise.png")

    # ablation comparison (): naive vs global neg-prompt vs OSRG
    save(hstrip([("naive (tight, no neg)", naive_full),
                 ("global neg-prompt", tuned_full),
                 ("OSRG (ours)", osrg_full)], cell_w=440), "16_ablation_compare.png")

    # 17) GHOST → FIX zoom-crop: tight crop of the removed region
    bb = mask_bbox(dil, pad=24)
    save(hstrip([("naive — ghost / re-insert", naive_full.crop(bb)),
                 ("global neg-prompt", tuned_full.crop(bb)),
                 ("OSRG (ours) — clean", osrg_full.crop(bb))], cell_w=360),
         "17_ghost_vs_fix.png")

    # 18) 4-stage pipeline strip
    save(hstrip([("1 · input (1 click)", orig),
                 ("2 · SAM mask", mask_overlay(orig, dil)),
                 ("3 · BLIP sees background", no_people),
                 ("4 · OSRG removal", osrg_full)], cell_w=430),
         "18_pipeline_strip.png")

    # before / after + storyboard
    save(hstrip([("BEFORE", orig), ("OURS · OSRG", osrg_full)], cell_w=560),
         "13_before_after.png")
    story = grid([
        ("00 original", orig),
        ("01 detect (green=keep)", draw_detections(orig, persons, main_idx)),
        ("05 removal mask (dilated)", mask_overlay(orig, dil)),
        ("10 naive (no neg)", naive_full),
        ("11 global neg-prompt", tuned_full),
        ("14 OSRG (ours)", osrg_full),
    ], cols=3, cell_w=460)
    save(story, "99_storyboard.png")

    # per-photo removal check: is a person still detected in the hole? (no GT needed)
    main_mask = persons[main_idx]["mask"]
    reins = {"naive":  reinsertion(naive_full, dil, main_mask),
             "global": reinsertion(tuned_full, dil, main_mask),
             "osrg":   reinsertion(osrg_full, dil, main_mask)}

    times["total"] = time.perf_counter() - t0
    print(f"  done in {times['total']:.1f}s — removed {len(removed)} | re-insert "
          f"naive={reins['naive']} global={reins['global']} osrg={reins['osrg']}")
    if DEVICE == "cuda": torch.cuda.empty_cache(); gc.collect()
    return {"stem": stem, "removed": len(removed), "result": tuned_full, "naive": naive_full,
            "osrg": osrg_full, "storyboard": story, "times": times, "reins": reins,
            "caption": bg, "persons": len(persons)}

## § 7 · Evaluation / Run on the 4 photos

In [ ]:
samples = load_photos()
results = [run_removal(s) for s in samples]

# stage-timing log (for the 'cheap GPU' / runtime slide)
import csv as _csv
tlog = [r for r in results if "times" in r]
if tlog:
    keys = ["detect", "sam_mask", "caption", "inpaint_osrg", "total"]
    with open("/kaggle/working/stage_timings.csv", "w", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=["stem"] + keys); w.writeheader()
        for r in tlog:
            w.writerow({"stem": r["stem"], **{k: round(r["times"].get(k, 0), 2) for k in keys}})
    print(f"GPU: {GPU_NAME} | dtype: {DTYPE}")
    print(f"{'stage':<14}{'avg seconds':>12}")
    for k in keys:
        print(f"{k:<14}{np.mean([r['times'].get(k,0) for r in tlog]):>12.2f}")

# per-photo removal comparison on OUR photos (re-insertion, no GT needed)
rome = [r for r in results if "reins" in r]
if rome:
    with open("/kaggle/working/rome_eval.csv", "w", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=["stem","persons","removed","reins_naive",
                                           "reins_global","reins_osrg","total_s","caption"]); w.writeheader()
        for r in rome:
            w.writerow({"stem": r["stem"], "persons": r["persons"], "removed": r["removed"],
                        "reins_naive": r["reins"]["naive"], "reins_global": r["reins"]["global"],
                        "reins_osrg": r["reins"]["osrg"], "total_s": round(r["times"]["total"],1),
                        "caption": r["caption"]})
    print("\nRe-insertion on our Rome photos (1 = a person is still there → removal failed):")
    print(f"{'variant':<10}{'fail rate':>10}")
    for v in ("naive", "global", "osrg"):
        print(f"{v:<10}{100*np.mean([r['reins'][v] for r in rome]):>9.0f}%")

# zip every step image for easy download into the slides
zip_path = "/kaggle/working/removal_steps.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(OUT_DIR.parent))
print("zipped ->", zip_path)
print("steps per photo saved under", OUT_DIR)

In [ ]:
# inline storyboards
for r in results:
    if "storyboard" in r:
        print("="*80, "\n", r["stem"], "— removed", r["removed"], "person(s)")
        plt.figure(figsize=(18, 12)); plt.imshow(r["storyboard"]); plt.axis("off"); plt.show()

In [ ]:
# ablation comparison for each photo: naive / global neg-prompt / OSRG
for r in results:
    if "osrg" in r:
        orig_img = samples[[s['stem'] for s in samples].index(r['stem'])]['image']
        ba = hstrip([("BEFORE", orig_img), ("naive", r["naive"]),
                     ("global neg", r["result"]), ("OSRG (ours)", r["osrg"])], cell_w=400)
        plt.figure(figsize=(22, 6)); plt.imshow(ba); plt.axis("off")
        plt.title(r["stem"]); plt.show()

## § 7b · Quantitative Ablation (proving the novelty)
> The Rome photos have **no ground-truth background**, so we build a *synthetic removal
> benchmark*: paste a COCO **person** (with its segmentation mask) onto another COCO image →
> the original image is the **true clean ground truth** for that region. We then measure each
> variant against that GT. The decisive comparison is **global negative-prompt vs OSRG**:
> it isolates the effect of the *spatial masking* (our novelty), not just the negative prompt.

In [ ]:
# Synthetic copy-paste removal benchmark (true GT)
import urllib.request, zipfile
from pycocotools.coco import COCO

def _coco():
    ann = Path("/tmp/instances_val2017.json")
    if not ann.exists():
        z = Path("/tmp/ann.zip")
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip", z)
        with zipfile.ZipFile(z) as f: f.extract("annotations/instances_val2017.json", "/tmp")
        Path("/tmp/annotations/instances_val2017.json").rename(ann)
    return COCO(str(ann))

def _fetch(info):
    p = Path("/tmp/coco_imgs") / info["file_name"]; p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists(): urllib.request.urlretrieve(info["coco_url"], p)
    return Image.open(p).convert("RGB")

def build_benchmark(n=BENCH_N, seed=SEED):
    coco = _coco(); rng = random.Random(seed)
    pcat = coco.getCatIds(catNms=["person"])[0]
    pimgs = coco.getImgIds(catIds=[pcat]); rng.shuffle(pimgs)
    bgs = coco.getImgIds(); rng.shuffle(bgs); bi = 0
    out = []
    for pid in pimgs:
        if len(out) >= n: break
        info = coco.loadImgs(pid)[0]; A = info["width"] * info["height"]
        anns = coco.loadAnns(coco.getAnnIds(imgIds=pid, catIds=[pcat], iscrowd=False))
        cand = [a for a in anns if 0.05 <= a["area"] / A <= 0.30]
        if not cand: continue
        ann = max(cand, key=lambda a: a["area"])
        try:
            person = _fetch(info).resize((PROC_W, PROC_H))
            bg = _fetch(coco.loadImgs(bgs[bi % len(bgs)])[0]).resize((PROC_W, PROC_H)); bi += 1
        except Exception: continue
        pm = coco.annToMask(ann).astype(np.uint8)
        pm = np.array(Image.fromarray(pm * 255).resize((PROC_W, PROC_H), Image.NEAREST))
        sel = pm > 127
        comp = np.array(bg).copy(); comp[sel] = np.array(person)[sel]
        out.append({"composite": Image.fromarray(comp), "gt": bg,
                    "mask": Image.fromarray((sel * 255).astype(np.uint8))})
    print(f"built {len(out)} synthetic removal pairs")
    return out

In [ ]:
# metrics
import lpips as _lpipslib
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim
_lpips = _lpipslib.LPIPS(net="alex").to(DEVICE).eval()

def _bbox(mask, pad=8):
    m = np.array(mask.convert("L")) > 127
    ys, xs = np.where(m)
    if len(xs) == 0: return (0, 0, mask.width, mask.height)
    return (max(0, xs.min()-pad), max(0, ys.min()-pad),
            min(mask.width, xs.max()+pad), min(mask.height, ys.max()+pad))

def m_lpips(a, b):
    fa = TF.to_tensor(a).unsqueeze(0).to(DEVICE).float()*2-1
    fb = TF.to_tensor(b).unsqueeze(0).to(DEVICE).float()*2-1
    with torch.no_grad(): return float(_lpips(fa, fb))

def m_reinserted(result, mask):
    m = np.array(mask.convert("L")) > 127; tot = max(1, int(m.sum()))
    for p in detect_persons(result, thr=0.5):
        if ((p["mask"] > 0) & m).sum() / tot > 0.10:
            return 1
    return 0

def variant_run(v, composite, mask):
    bgp = BG_PROMPT_TMPL.format(bg="background scene")
    if v == "sd_baseline":
        return inpaint(composite, mask, NAIVE_PROMPT, "")[0]
    if v == "global_neg":
        return inpaint(composite, mask, bgp, REMOVAL_NEG)[0]
    return osrg_inpaint(composite, mask, bgp, OBJ_PROMPT)[0]   # ours

In [ ]:
# run the ablation (+ benchmark explainer + 2AFC user-study pairs)
VARIANTS = ["sd_baseline", "global_neg", "osrg"]
bench = build_benchmark(BENCH_N)
US_DIR = Path("/kaggle/working/user_study"); US_DIR.mkdir(parents=True, exist_ok=True)
abl_rows, us_key = [], []
rng_us = random.Random(SEED)
for k, s in enumerate(bench):
    bb = _bbox(s["mask"]); gtc = s["gt"].crop(bb); res_by = {}
    for v in VARIANTS:
        res = variant_run(v, s["composite"], s["mask"]); res_by[v] = res
        rc = res.crop(bb); a, b = np.array(rc), np.array(gtc)
        abl_rows.append({"i": k, "variant": v,
            "lpips": round(m_lpips(rc, gtc), 4),
            "psnr": round(float(_psnr(b, a, data_range=255)), 3),
            "ssim": round(float(_ssim(b, a, channel_axis=2, data_range=255)), 4),
            "reins": m_reinserted(res, s["mask"])})

    # benchmark explainer (first sample): composite → mask → GT → OSRG result
    if k == 0:
        hstrip([("composite (object pasted in)", s["composite"]),
                ("removal mask", mask_overlay(s["composite"], np.array(s["mask"].convert("L")))),
                ("ground truth (clean)", s["gt"]),
                ("OSRG result", res_by["osrg"])], cell_w=380).save("/kaggle/working/benchmark_example.png")

    # 2AFC user-study pair: OSRG vs global_neg, sides randomised
    flip = rng_us.random() < 0.5
    A, B = (res_by["osrg"], res_by["global_neg"]) if not flip else (res_by["global_neg"], res_by["osrg"])
    pair = Image.new("RGB", (PROC_W*2+8, PROC_H), (255,255,255))
    pair.paste(A,(0,0)); pair.paste(B,(PROC_W+8,0)); pair.save(US_DIR/f"pair_{k:02d}.png")
    us_key.append({"pair":k, "left":"osrg" if not flip else "global_neg",
                   "right":"global_neg" if not flip else "osrg"})
    print(f"  benchmark {k+1}/{len(bench)} done")

(US_DIR/"key.json").write_text(json.dumps(us_key, indent=2))
print(f"user-study: {len(us_key)} 2AFC pairs -> {US_DIR} (ask viewers 'which looks more realistic?')")

with open("/kaggle/working/ablation.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["i","variant","lpips","psnr","ssim","reins"]); w.writeheader()
    w.writerows(abl_rows)

# aggregate
agg = {v: {"lpips":[], "psnr":[], "ssim":[], "reins":[]} for v in VARIANTS}
for r in abl_rows:
    for k in ("lpips","psnr","ssim","reins"): agg[r["variant"]][k].append(r[k])
print(f"\n{'variant':<14}{'LPIPS↓':>9}{'PSNR↑':>9}{'SSIM↑':>9}{'Re-insert↓':>12}")
print("-"*53)
for v in VARIANTS:
    a = agg[v]; n = len(a["lpips"])
    print(f"{v:<14}{np.mean(a['lpips']):>9.4f}{np.mean(a['psnr']):>9.2f}"
          f"{np.mean(a['ssim']):>9.4f}{100*np.mean(a['reins']):>11.1f}%")
print("\nKey comparison: global_neg vs osrg isolates the spatial-masking novelty.")

In [ ]:
# ablation bar chart
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for j, (k, lbl, better) in enumerate([("lpips","LPIPS (lower better)","↓"),
                                       ("psnr","PSNR (higher better)","↑"),
                                       ("ssim","SSIM (higher better)","↑"),
                                       ("reins","Re-insertion rate","↓")]):
    vals = [np.mean(agg[v][k]) * (100 if k=="reins" else 1) for v in VARIANTS]
    ax[j].bar(VARIANTS, vals, color=["#bbb","#88b","#3a7"]); ax[j].set_title(lbl)
    ax[j].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("/kaggle/working/ablation_chart.png", dpi=120); plt.show()
print("saved ablation.csv + ablation_chart.png")